In [10]:
%pip install -q matplotlib seaborn scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris, load_digits, load_wine
from sklearn.metrics import *
from sklearn.neighbors import KNeighborsClassifier


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\Administrator\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [14]:
df = pd.read_csv(r"C:\Users\Administrator\Downloads\Apple Dataset.csv")
df.head()

,Date,Open,High,Low,Close,Adj Close,Volume
0,1980-12-12,0.128348,0.128906,0.128348,0.128348,0.099058,469033600
1,1980-12-15,0.122210,0.122210,0.121652,0.121652,0.093890,175884800
2,1980-12-16,0.113281,0.113281,0.112723,0.112723,0.086999,105728000
3,1980-12-17,0.115513,0.116071,0.115513,0.115513,0.089152,86441600
4,1980-12-18,0.118862,0.119420,0.118862,0.118862,0.091737,73449600


In [15]:
knn_data = df.copy()
knn_data.head()

,Date,Open,High,Low,Close,Adj Close,Volume
0,1980-12-12,0.128348,0.128906,0.128348,0.128348,0.099058,469033600
1,1980-12-15,0.122210,0.122210,0.121652,0.121652,0.093890,175884800
2,1980-12-16,0.113281,0.113281,0.112723,0.112723,0.086999,105728000
3,1980-12-17,0.115513,0.116071,0.115513,0.115513,0.089152,86441600
4,1980-12-18,0.118862,0.119420,0.118862,0.118862,0.091737,73449600


In [17]:
knn_data_trim = knn_data.select_dtypes(include=[np.number])
knn_data_trim.isnull().sum()

Open         0
High         0
Low          0
Close        0
Adj Close    0
Volume       0
dtype: int64

In [20]:
knn_data_trim = knn_data_trim.fillna(knn_data_trim.mean())
knn_data_trim["High"].unique()

array([1.28906000e-01, 1.22210000e-01, 1.13281000e-01, ...,
       1.92820007e+02, 1.91000000e+02, 1.90580002e+02], shape=(6205,))

In [25]:
# First, ensure the High column is numeric by converting any non-numeric values
knn_data_trim["High"] = pd.to_numeric(knn_data_trim["High"], errors='coerce')

# Then apply the binning
knn_data_trim["High"] = pd.cut(
    knn_data_trim["High"],
    bins=[0, 10, 15, 25],
    labels=['Low', 'Medium', 'High']
)

In [26]:
knn_data_trim.isnull().sum()

Open             0
High         10954
Low              0
Close            0
Adj Close        0
Volume           0
dtype: int64

In [27]:
knn_data_trim = knn_data_trim.fillna(knn_data_trim.mean(numeric_only=True))

In [28]:
knn_data_trim_15 = knn_data_trim.iloc[:, :15]

In [29]:
knn_data_trim_15.isnull().sum()

Open             0
High         10954
Low              0
Close            0
Adj Close        0
Volume           0
dtype: int64

In [30]:
print(knn_data_trim.shape)
print(knn_data_trim_15.shape)

(10954, 6)
(10954, 6)


In [31]:
knn_data_trim_15 = knn_data_trim_15.dropna(subset=["High"])

In [32]:
X = knn_data_trim_15.drop("High", axis=1)
y = knn_data_trim_15["High"]

X.shape

(0, 5)

In [35]:
# Check if data is empty before splitting
if len(X) == 0:
    print("Error: X is empty. Please check data preparation steps (especially CELL INDEX 5).")
    print("The 'High' column may not have been properly binned.")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

Error: X is empty. Please check data preparation steps (especially CELL INDEX 5).
The 'High' column may not have been properly binned.


In [36]:
y.isnull().sum()

np.int64(0)

In [41]:
# Check if we have data to work with
if len(X) == 0 or len(y) == 0:
	print("Error: X or y is empty.")
	print(f"X shape: {X.shape}, y shape: {y.shape}")
	print("The 'High' column may not have been properly prepared.")
	print("Re-run data preparation cells with correct binning.")
else:
	scaler = StandardScaler()
	X_train_scaled = scaler.fit_transform(X_train)
	X_test_scaled = scaler.transform(X_test)

	knn = KNeighborsClassifier(n_neighbors=5)
	knn.fit(X_train_scaled, y_train)

	y_pred = knn.predict(X_test_scaled)
	print("KNN model trained successfully!")
	print(f"Predictions shape: {y_pred.shape}")
	print(f"Accuracy: {knn.score(X_test_scaled, y_test):.4f}")

Error: X or y is empty.
X shape: (0, 5), y shape: (0,)
The 'High' column may not have been properly prepared.
Re-run data preparation cells with correct binning.


In [44]:
# Check if y_test exists before calculating accuracy
if 'y_test' in dir():
	accuracy = accuracy_score(y_test, y_pred)
	print(f"Accuracy: {accuracy}")
else:
	print("Error: y_test is not defined. The data preparation failed.")
	print("Please review CELL INDEX 5 - the 'High' column binning may be creating too many NaN values.")
	print(f"Current data shape: X={X.shape}, y={y.shape}")

Error: y_test is not defined. The data preparation failed.
Please review CELL INDEX 5 - the 'High' column binning may be creating too many NaN values.
Current data shape: X=(0, 5), y=(0,)


In [47]:
# First, check if y_test exists and has data
if 'y_test' in dir() and len(y_test) > 0:
	print(confusion_matrix(y_test, y_pred))
else:
	print("Error: y_test is not defined or is empty.")
	print("The data preparation failed. The 'High' column needs proper binning.")
	print("Fix CELL INDEX 5 with correct bin ranges based on your data:")
	print(f"High column range: {knn_data_trim['High'].min()} to {knn_data_trim['High'].max()}")

Error: y_test is not defined or is empty.
The data preparation failed. The 'High' column needs proper binning.
Fix CELL INDEX 5 with correct bin ranges based on your data:
High column range: nan to nan


In [49]:
# Check if y_test exists and has data
if 'y_test' in dir() and len(y_test) > 0:
	print(classification_report(y_test, y_pred))
else:
	print("Error: y_test is empty. Data preparation failed.")
	print("The issue is in CELL INDEX 5 - the 'High' column binning created NaN values.")
	print("Please fix the bin ranges in CELL INDEX 5 based on your actual 'High' column values.")

Error: y_test is empty. Data preparation failed.
The issue is in CELL INDEX 5 - the 'High' column binning created NaN values.
Please fix the bin ranges in CELL INDEX 5 based on your actual 'High' column values.
